In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from Data_Analysis import *


In [ ]:
url = "https://community.linkareer.com/mentoring"
html = requests.get(url)
soup = BeautifulSoup(html.content, "html.parser")

In [ ]:
soup

In [ ]:
tr_tags_with_data_key = soup.select('tr[data-row-key]')

data_row_keys_css = []
for tr in tr_tags_with_data_key:
    data_row_keys_css.append(tr['data-row-key'])

print(data_row_keys_css)

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mentoring?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_mentoring_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
for key in data_row_keys_css:
    url = f"https://community.linkareer.com/mentoring/{key}?page=1"
    html = requests.get(url)
    soup = BeautifulSoup(html.content, "html.parser")
    title = soup.find('h1', class_='header-title')
    content = soup.find('div', class_='post-detail').text.strip()
    comments = soup.find_all('li', class_='comment')
    if comments:
        for comment in comments:
            scores = comment.find_all('div', class_='sc-7450030d-0 geJdfg').find('span')
            high_score = max(scores)




In [ ]:
url = "https://community.linkareer.com/mentoring/4422329?page=1"
html = requests.get(url)
soup = BeautifulSoup(html.content, "html5lib")
title = soup.find('h1', class_='header-title').text


In [ ]:
title

In [ ]:
content = soup.find('div', class_='post-detail').text.strip()
content

In [ ]:
main_div = soup.find('div', class_='content')
main_div

In [ ]:
# ① 댓글 전체가 들어있는 컨테이너
comments_section = soup.find(id="post-detail-comments-section")

# ② 각 댓글 블록(예: .form-comment-root 아래 개별 댓글)
#    실제 클래스 이름은 페이지 구조에 맞게 고쳐주세요.
comments = comments_section.select("div.form-comment-root div.editor-box")

for idx, comment in enumerate(comments, 1):
    text = comment.get_text(strip=True)
    print(f"{idx}. {text}")

In [ ]:
soup

In [100]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mentoring/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_mentoring_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_QA.json', orient='records', force_ascii=False, indent=4)


Scraping posts:  33%|███▎      | 1/3 [00:06<00:12,  6.49s/post]

Post ID: 4423268, Title: 인턴 서류평가시 가장 중요하게 평가되는 항목, Best Rate: 56.0, Best Comment: 안녕하세요. 인턴 서류에서 가장 중요하게 평가되는 항목은 지원하는 회사와 직무에 따라 조금...


Scraping posts:  67%|██████▋   | 2/3 [00:12<00:06,  6.11s/post]

Post ID: 4423041, Title: 영업교육 경력을 기재하는게 좋을지 빼는게 좋을지 고민입니다., Best Rate: 55.0, Best Comment: 저라면 쓸거같아요. 왜냐하면 영업관리직무 지원하면 점포,매장관리 및 교육을 기본적으로 하거...


Scraping posts: 100%|██████████| 3/3 [00:17<00:00,  5.98s/post]

Post ID: 4422416, Title: 면접에서 지원한 직무랑 안어울린다는 말을 들었어요 .., Best Rate: 56.0, Best Comment: 안녕하세요. 면접관도 사람인지라, 짧은 시간 안에 지원자를 완벽하게 파악하기는 어렵습니다....


In [97]:
with open('linkareer_mentoring_data_row_keys.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

post_ids = [int(d['0']) for d in raw]

# post_ids각 요소들의 뒷 두자리는 삭제
real_ids = [int(str(pid)[:-2]) for pid in post_ids]

real_ids

[4423268,
 4423041,
 4422416,
 4422329,
 4422327,
 4422165,
 4422096,
 4421777,
 4421759,
 4421543,
 4421540,
 4421509,
 4421500,
 4421231,
 4421197,
 4421013,
 4421006,
 4420926,
 4420638,
 4420635,
 4420575,
 4419960,
 4419613,
 4419123,
 4419096,
 4418821,
 4418203,
 4416766,
 4416561,
 4416513,
 4415867,
 4415825,
 4415805,
 4415787,
 4415282,
 4415251,
 4415119,
 4415025,
 4414655,
 4414501,
 4414335,
 4413934,
 4412881,
 4412473,
 4411893,
 4411571,
 4410831,
 4410324,
 4410265,
 4409814,
 4409704,
 4409245,
 4409225,
 4409207,
 4409155,
 4408796,
 4408410,
 4408187,
 4408138,
 4408037,
 4407517,
 4407507,
 4407476,
 4407475,
 4407295,
 4406446,
 4406429,
 4404667,
 4404661,
 4404386,
 4404342,
 4404240,
 4404091,
 4404018,
 4404014,
 4403951,
 4403893,
 4403850,
 4403540,
 4403398,
 4403385,
 4403273,
 4403194,
 4403140,
 4402760,
 4402582,
 4402467,
 4402314,
 4401557,
 4401352,
 4401241,
 4401025,
 4400849,
 4400353,
 4400244,
 4400227,
 4400167,
 4400003,
 4399911,
 4399559,


In [92]:
chrome_opts = Options()
chrome_opts.add_argument("--headless")
chrome_opts.add_argument("--no-sandbox")
chrome_opts.add_argument("--disable-dev-shm-usage")
chrome_opts.add_argument("--window-size=1920,1080")
chrome_opts.binary_location = (
    "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
)

# 2) Homebrew로 설치한 Intel용 드라이버 경로 지정
service = Service(executable_path="/opt/homebrew/bin/chromedriver")

# 3) 드라이버 실행
driver = webdriver.Chrome(service=service, options=chrome_opts)

url = f"https://community.linkareer.com/mentoring/4422329?page=1"
driver.get(url)
time.sleep(5)  # 필요에 따라 WebDriverWait 로 변경 가능

try:
    title = driver.find_element(
        By.CSS_SELECTOR,
        "#post-detail-container .header-title"
    ).text.strip()
except:
    title = ""

    # 2) 정확한 본문 셀렉터
try:
    content = driver.find_element(
        By.CSS_SELECTOR,
        "#post-detail-content-container .post-detail"
    ).text.strip()
except:
    content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
best_rate = -1.0
best_comment = ""
for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
    try:
        # (이 부분도 실제 클래스명에 맞게 변경)
        rate_txt = li.find_element(
            By.CSS_SELECTOR,
            ".sc-7450030d-0.geJdfg p"
        ).text
        rate = float(re.search(r'(\d+)', rate_txt).group(1))
        if rate > best_rate:
            best_rate = rate
            best_comment = li.find_element(
                By.CSS_SELECTOR,
                ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
            ).text.strip()
    except:
        continue
print(f"Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

Title: 00년생 여자 취업 루트 조언 부탁드입니다.., Best Rate: 56.0, Best Comment: 안녕하세요. 현재 시점부터 하반기 공채(9월)까지 남은 약 3개월의 시간을 고려했을 때, ...


In [90]:
title

'00년생 여자 취업 루트 조언 부탁드입니다..'

In [71]:
content_el = driver.find_element(
    By.CSS_SELECTOR,
    "div.editor-viewer.post-detail.sc-35fc92de-0.fdeRQs div.post-detail"
)
content = content_el.text.strip()

In [72]:
content

'해외대라 작년 7월에 졸업해서 2024년 하반기 공채 바로 준비했었고, 그 때 초심자의 행운이었던 건지 대기업 최종면접까지 갔고 결국에는 떨어졌습니다\n  이후 공백기 늘리고 싶지 않아 하반기 공채 끝난 후 12월부터 올해 6월까지 대기업 광고대행사 한 곳이서 6개월 동안 인턴했고 지금은 인턴 직무가 저와 맞지 않아 퇴사하게 되었습니다.\n  올해 상반기 공채도 인턴하면서 같이 준비했는데 작년 하반기 보다 서류 합격률은 높았지만 1차면접 까지 간 건 한 군데 뿐이었고 결국 상반기 공채도 다 떨어졌습니다ㅠㅠ\n  하반기 공채가 보통 9월에 열리니 남은 시간까지 제가 하고 싶은 직무 관련 인턴 하나 더 하고 싶은데\n요즘 인턴 구하기더 만만치 않은 것 같아요...\n  전공은 방송 관련 전공이라 pd 직군 아니면 사기업 마케팅 (콘텐츠 제작) 위주로 직무 생각하고 있는데 현재 6월부터 앞으로 취준 방향을 어떻게 해야할지 조언 부탁드립니다... 인턴을 어떻게든 구하는게\n좋을지 작은 스타트업 회사 정규직으로라도 들어가야할지 아니면 하반기 공채 전까지 사이드프로젝트+휴식 쪽으로 가야할지 조언부탁드랴요..!!\n  00년생 여자이고, 주변 친구들은 다 취업해서 회사다니는데 전 언제 취업할 수 있을건지 걱정이네요ㅠㅠㅠ'

In [73]:
items = driver.find_elements(
    By.CSS_SELECTOR,
    "#post-detail-comments-section ul li"
)
best_rate = -1.0
best_comment = ""
for li in items:
    try:
        rate_p = li.find_element(
            By.CSS_SELECTOR,
            ".sc-7450030d-0.geJdfg p"
        )
        m = re.search(r'(\d+(?:\.\d+)?)\%', rate_p.text)
        rate = float(m.group(1)) if m else 0.0
        if rate > best_rate:
            best_rate = rate
            best_comment = li.find_element(
                By.CSS_SELECTOR,
                ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
            ).text.strip()
    except Exception:
        continue

In [75]:
best_comment

"안녕하세요. 현재 시점부터 하반기 공채(9월)까지 남은 약 3개월의 시간을 고려했을 때, \n희망 직무와 연관된 실질적인 경험을 더 쌓는 것입니다. 지난 6개월의 인턴 경험이 직무와 맞지 않으셨다고 하니, \n이번에는 꼭 PD 또는 마케팅(콘텐츠 제작)과 관련된 인턴십을 찾아보시는 게 좋을 것 같습니다.\n인턴십 지원 시에 해당 직무에 대한 이해도와 본인의 강점을 명확히 어필하는 것이 중요하구요.\n지난 인턴 경험에서 얻은 인사이트를 녹여내어 '왜 이 직무가 나에게 더 맞는지'를 설명할 수 있다면 좋습니다.\n주로 면접에서 미끌어진다면 모의 면접이나 면접스터디등을 통해 면접스킬을 키우시길 추천드립니다."

56.0